# CDT Quantification Error Analysis

Boxplots of per-batch **quantification Absolute Error (AE)** for every method, with a
dropdown to switch between datasets. Each box is one method, colored by its family:

- 🔴 **syn** (synthetic, `*_syn` + `DySyn`)
- 🟢 **cdt** (CDT-gated, `*_cdt`)
- 🔵 **classic** (binary OvR quantifiers)
- 🟡 **multiclass** (native multiclass quantifiers)

AE for a batch = mean over classes of `|normalized_prediction - real_prevalence|`.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# --- Datasets to plot (label -> results CSV) ---
DATASETS = {
    "Dermatology": "cdt_results/Dermatology/Dermatology_results.csv",
    "Nursery": "cdt_results/Nursery/Nursery_results.csv",
}

# Native multiclass quantifiers (see get_multiclass_quantifier_map in ovr.py)
MULTICLASS = {"GAC", "GPAC", "EMQ", "KDEyHD", "KDEyCS", "KDEyML", "FM", "HDx", "PWK", "CC2"}

# Category -> color and left-to-right grouping order on the x-axis
CATEGORY_COLORS = {
    "classic": "royalblue",
    "cdt": "seagreen",
    "syn": "crimson",
    "multiclass": "gold",
}
CATEGORY_ORDER = ["classic", "cdt", "syn", "multiclass"]


def categorize(qnt: str) -> str:
    """Map a quantifier name to one of: classic / cdt / syn / multiclass."""
    if qnt.endswith("_cdt"):
        return "cdt"
    if qnt.endswith("_syn") or qnt == "DySyn":  # DySyn is the synthetic DyS variant
        return "syn"
    if qnt in MULTICLASS:
        return "multiclass"
    return "classic"


def load_results(path: str) -> pd.DataFrame:
    """Load a *_results.csv and compute per-batch Absolute Error for each method."""
    df = pd.read_csv(path)
    norm_cols = [c for c in df.columns if c.endswith("_p_normalized")]
    real_cols = [c[: -len("_p_normalized")] + "_real" for c in norm_cols]
    df["error"] = np.abs(df[norm_cols].values - df[real_cols].values).mean(axis=1)
    df["category"] = df["qnt"].map(categorize)
    return df[["qnt", "category", "error"]]


data = {name: load_results(path) for name, path in DATASETS.items()}
for name, df in data.items():
    print(f"{name}: {df['qnt'].nunique()} methods, {len(df)} rows")

Dermatology: 42 methods, 42000 rows
Nursery: 42 methods, 42000 rows


In [2]:
fig = go.Figure()
dataset_names = list(data.keys())

# Always-visible, data-free traces just to provide a stable 4-entry color legend
for cat in CATEGORY_ORDER:
    fig.add_trace(go.Box(
        y=[None], name=cat, marker_color=CATEGORY_COLORS[cat],
        legendgroup=cat, showlegend=True, visible=True, hoverinfo="skip",
    ))
trace_dataset = [None] * len(CATEGORY_ORDER)  # legend traces belong to no dataset

# One box per method per dataset; only the first dataset's boxes start visible
for ds_idx, ds_name in enumerate(dataset_names):
    df = data[ds_name]
    methods = sorted(df["qnt"].unique(),
                     key=lambda q: (CATEGORY_ORDER.index(categorize(q)), q))
    for q in methods:
        cat = categorize(q)
        fig.add_trace(go.Box(
            y=df.loc[df["qnt"] == q, "error"],
            name=q, marker_color=CATEGORY_COLORS[cat],
            legendgroup=cat, showlegend=False,
            visible=(ds_idx == 0), boxpoints="outliers",
        ))
        trace_dataset.append(ds_name)

# Dropdown: show only the selected dataset's boxes (legend traces stay visible)
buttons = []
for ds_name in dataset_names:
    vis = [td is None or td == ds_name for td in trace_dataset]
    buttons.append(dict(
        label=ds_name, method="update",
        args=[{"visible": vis},
              {"title.text": f"Quantification Absolute Error by Method — {ds_name}"}],
    ))

fig.update_layout(
    updatemenus=[dict(buttons=buttons, direction="down", showactive=True,
                      x=1.0, xanchor="right", y=1.18, yanchor="top")],
    title=dict(text=f"Quantification Absolute Error by Method — {dataset_names[0]}"),
    xaxis_title="Method", yaxis_title="Absolute Error (mean over classes)",
    legend_title_text="Family", template="plotly_white", height=600,
)
fig.update_xaxes(tickangle=-45)
fig.show()

# CDT Distance Distribution Analysis

Distribution of the per-batch **DyS distances** collected inside each CDT during
`fit()` — the values used to set the drift threshold `thr = mean + 2·std` — as saved
to `cdt_results/<dataset>/distances.csv` (one binary one-vs-rest model per row).

- **Dataset** dropdown — choose which dataset's distances to inspect.
- **Model** dropdown — choose a single binary model, or **All models** to overlay
  every model together (one color each).

Distances span many orders of magnitude (median ≈ 1e-12, with a thin tail up to
≈ 1e-3), so they are plotted on a **log₁₀** x-axis.

In [3]:
import os
import glob
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

# Discover every dataset that has a CDT distances file
DISTANCE_FILES = {
    os.path.basename(os.path.dirname(p)): p
    for p in sorted(glob.glob("cdt_results/*/distances.csv"))
}


def load_distances(path):
    """Return ({model_id: distances}, {model_id: thr}) for one dataset.

    Each row of distances.csv is one binary (one-vs-rest) model, carrying its
    DyS distances and the drift threshold `thr = mean + 2·std` fitted from them.
    We group by model_id (keeping the last row) so that, if a file still carries
    duplicate rows from an earlier run, only one series per model is kept.
    """
    df = pd.read_csv(path).drop_duplicates(subset="model_id", keep="last")
    distances, thresholds = {}, {}
    for r in df.itertuples(index=False):
        model = str(r.model_id)
        distances[model] = np.asarray(json.loads(r.distances), dtype=float)
        thresholds[model] = float(r.thr)
    return distances, thresholds


_loaded = {name: load_distances(path) for name, path in DISTANCE_FILES.items()}
DISTANCES = {name: dist for name, (dist, _thr) in _loaded.items()}
THRESHOLDS = {name: thr for name, (_dist, thr) in _loaded.items()}
for name, models in DISTANCES.items():
    print(f"{name}: {len(models)} binary models -> {list(models)}")

Avila: 12 binary models -> ['G', 'A', 'H', 'F', 'E', 'I', 'W', 'X', 'D', 'Y', 'C', 'B']
Chessgame: 18 binary models -> ['fourteen', 'twelve', 'nine', 'ten', 'eleven', 'fifteen', 'thirteen', 'five', 'draw', 'six', 'seven', 'eight', 'two', 'four', 'three', 'one', 'sixteen', 'zero']
Dermatology: 6 binary models -> ['2', '4', '5', '3', '1', '6']
Land-use: 8 binary models -> ['Corn', 'Soybeans', 'Alfalfa', 'Trees', 'Hay', 'Grass', 'Wheat', 'Oats']
Mfeat: 1 binary models -> ['8']
Nursery: 4 binary models -> ['spec_prior', 'not_recom', 'priority', 'very_recom']


In [4]:
# --- Avila: stored thr vs. recomputed thr (mean + 2·std) -------------------
# The CDT computes its drift threshold in RAW distance space (see
# methods/quantifiers_utils.py): thr = mean(distances) + 2·std(distances), with
# np.std (population std, ddof=0). The curve above is drawn in log10 space, so a
# heavy-tailed distance distribution pushes mean+2·std far right of the peak.
# Here we re-derive thr from the saved distances to confirm it matches the
# stored value, and show where the threshold falls relative to the bulk.
avila = DISTANCES["Avila"]

rows = []
for model_id, dist in avila.items():
    recomputed = float(np.mean(dist) + 2 * np.std(dist))  # ddof=0, matches CDT
    stored = THRESHOLDS["Avila"][model_id]
    rows.append({
        "model_id": model_id,
        "n": dist.size,
        "thr_stored": stored,
        "thr_recomputed": recomputed,
        "match": np.isclose(stored, recomputed, rtol=1e-9),
        "median": float(np.median(dist)),
        "mean": float(np.mean(dist)),
        "std": float(np.std(dist)),
        "max": float(np.max(dist)),
        # how many std-devs past the median the threshold sits (skew indicator)
        "thr_vs_median_in_std": float((stored - np.median(dist)) / (np.std(dist) or np.nan)),
    })

avila_thr = pd.DataFrame(rows).sort_values("thr_stored", ascending=False)
print("All thresholds match recompute:", bool(avila_thr["match"].all()))
avila_thr

All thresholds match recompute: True


,model_id,n,thr_stored,thr_recomputed,match,median,mean,std,max,thr_vs_median_in_std
11,B,1000,1.165637e-01,1.165637e-01,True,4.843287e-13,5.678115e-03,5.544280e-02,5.572275e-01,2.102414
6,W,1000,1.275339e-02,1.275339e-02,True,1.972701e-12,6.102173e-04,6.071585e-03,6.102173e-02,2.100504
9,Y,1000,3.571058e-03,3.571058e-03,True,7.506926e-13,1.708661e-04,1.700096e-03,1.708661e-02,2.100504
7,X,1000,1.076976e-03,1.076976e-03,True,8.116499e-13,5.312229e-05,5.119269e-04,5.144292e-03,2.103769
8,D,1000,8.701353e-04,8.701353e-04,True,9.518673e-13,4.163377e-05,4.142508e-04,4.163377e-03,2.100504
4,E,1000,7.367261e-04,7.367261e-04,True,8.381151e-13,4.614262e-05,3.452917e-04,3.303048e-03,2.133634
3,F,1000,2.651399e-04,2.651399e-04,True,7.056194e-13,1.804845e-05,1.235457e-04,1.144665e-03,2.146087
5,I,1000,2.340863e-04,2.340863e-04,True,6.427390e-13,1.120045e-05,1.114429e-04,1.120044e-03,2.100504
2,H,1000,2.229084e-04,2.229084e-04,True,6.934342e-13,1.100714e-05,1.059506e-04,1.064163e-03,2.103889
1,A,1000,5.024114e-05,5.024114e-05,True,8.474687e-13,1.971820e-06,2.413466e-05,6.147277e-04,2.081701


In [5]:
from scipy.stats import gaussian_kde

# Distances span many orders of magnitude (median ~1e-12, tail up to ~1e-3), so
# they are shown on a log10 axis. Non-positive values (floating-point noise near
# 0) are clipped to a small floor before taking the log.
LOG_FLOOR = 1e-15
ALL_LABEL = "All models"
GRID_POINTS = 300
PALETTE = px.colors.qualitative.Plotly


def log_distances(values):
    return np.log10(np.clip(values, LOG_FLOOR, None))


def log_grid(dataset):
    """Common log10 x-grid so the density curves are directly comparable."""
    allv = np.concatenate([log_distances(v) for v in DISTANCES[dataset].values()])
    lo, hi = allv.min(), allv.max()
    pad = 0.05 * (hi - lo or 1.0)
    return np.linspace(lo - pad, hi + pad, GRID_POINTS)


def kde_density(logd, grid):
    """Gaussian-KDE density on `grid`; jitter guards near-zero-variance series."""
    if np.std(logd) < 1e-9:
        logd = logd + np.random.default_rng(0).normal(0, 1e-6, size=logd.size)
    return gaussian_kde(logd)(grid)


def model_color(dataset, model):
    names = list(DISTANCES[dataset])
    return PALETTE[names.index(model) % len(PALETTE)]


def make_traces(dataset, selection):
    models = DISTANCES[dataset]
    names = list(models) if selection == ALL_LABEL else [selection]
    grid = log_grid(dataset)
    return [
        go.Scatter(
            x=grid,
            y=kde_density(log_distances(models[name]), grid),
            mode="lines",
            name=str(name),
            line=dict(color=model_color(dataset, name), width=2.5),
        )
        for name in names
    ]


def add_threshold_lines(fig, dataset, selection):
    """Draw each model's drift threshold `thr` as a dashed vertical line, on the
    same log10 axis as the distance curves and colored to match each curve."""
    models = DISTANCES[dataset]
    names = list(models) if selection == ALL_LABEL else [selection]
    for name in names:
        thr_log = float(log_distances(np.asarray([THRESHOLDS[dataset][name]]))[0])
        fig.add_vline(
            x=thr_log,
            line=dict(color=model_color(dataset, name), width=1.5, dash="dash"),
            annotation_text=f"thr {name}",
            annotation_position="top",
            annotation_font_color=model_color(dataset, name),
        )


def make_figure(dataset, selection):
    fig = go.Figure()
    for trace in make_traces(dataset, selection):
        fig.add_trace(trace)
    add_threshold_lines(fig, dataset, selection)
    sel = "all models" if selection == ALL_LABEL else f"model {selection}"
    fig.update_layout(
        template="plotly_white", height=520,
        xaxis_title="log₁₀(DyS distance)", yaxis_title="Density",
        legend_title_text="Binary model",
        title=f"CDT DyS distance distribution — {dataset} ({sel})",
    )
    return fig


# --- Dropdowns: dataset, then model (with an "all models" option) ---
default_ds = "Nursery" if "Nursery" in DISTANCES else next(iter(DISTANCES))
dataset_dd = widgets.Dropdown(
    options=list(DISTANCES), value=default_ds, description="Dataset:",
)
model_dd = widgets.Dropdown(
    options=[ALL_LABEL] + list(DISTANCES[default_ds]),
    value=ALL_LABEL, description="Model:",
)

# A single persistent figure updated in place — avoids stacking a new plot on
# every dropdown change (the failure mode of re-calling fig.show()).
fig_widget = go.FigureWidget()


def redraw(*_):
    src = make_figure(dataset_dd.value, model_dd.value)
    with fig_widget.batch_update():
        fig_widget.data = []  # drop previous traces
        for trace in src.data:
            fig_widget.add_trace(trace)
        fig_widget.layout.shapes = src.layout.shapes        # threshold lines
        fig_widget.layout.annotations = src.layout.annotations
        fig_widget.layout.update(src.layout, overwrite=False)


def on_dataset_change(_):
    # Repopulate the model dropdown for the newly selected dataset, then redraw.
    model_dd.unobserve(redraw, names="value")
    model_dd.options = [ALL_LABEL] + list(DISTANCES[dataset_dd.value])
    model_dd.value = ALL_LABEL
    model_dd.observe(redraw, names="value")
    redraw()


dataset_dd.observe(on_dataset_change, names="value")
model_dd.observe(redraw, names="value")

redraw()
display(widgets.VBox([widgets.HBox([dataset_dd, model_dd]), fig_widget]))